In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# ============================================================
# 데이터 불러오기
# ============================================================

from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

train_full = pd.read_parquet(PROJECT_ROOT / 'data' / '1_train_full.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'data' / '1_test.parquet')

## 1. 피처 엔지니어링(파생변수 생성)

In [ ]:
# ============================================================
# 1-1. 수위 차분 파생변수 생성
# ============================================================
# wl_diff_00h = wl_t(현재수위) - wl_t_minus_00h 파생변수 5개 생성

print('train 파생변수 추가 전 크기: ', train_full.shape)
print('test 파생변수 추가 전 크기: ', test.shape)

for h in [1, 3, 6, 12, 24]:
    train_full[f"wl_diff_{h}h"] = (
        train_full['wl_t'] - train_full[f"wl_t_minus_{h}h"]
    )
    test[f"wl_diff_{h}h"] = (
        test['wl_t'] - test[f"wl_t_minus_{h}h"]
    )

print('train 수위차분 파생변수 추가 후 크기: ', train_full.shape)
print('test 수위차분 파생변수 추가 후 크기: ', test.shape)


In [ ]:
# ============================================================
# 1-2. 강우강도 파생변수 생성
# ============================================================
# imerg_int_0to0h = (imerg_rain_00h - imerg_rain_00h)/hours
# sfc_int_0to0h = (sfc_rain_00h - sfc_rain_00h)/hours

print('train 파생변수 추가 전 크기: ', train_full.shape)
print('test 파생변수 추가 전 크기: ', test.shape)


for df in [train_full, test]:

    df['imerg_int_1to6h'] = (df['imerg_rain_6h'] - df['imerg_rain_1h']) / 5
    df['imerg_int_6to24h'] = (df['imerg_rain_24h'] - df['imerg_rain_6h']) / 18

    df['sfc_int_1to6h'] = (df['sfc_rain_6h'] - df['sfc_rain_1h']) / 5
    df['sfc_int_6to24h'] = (df['sfc_rain_24h'] - df['sfc_rain_6h']) / 18


print('train 강우강도 파생변수 추가 후 크기: ', train_full.shape)
print('test 강우강도 파생변수 추가 후 크기: ', test.shape)

In [ ]:
# ============================================================
# 1-3. 최종 컬럼군 정의
# ============================================================

# 종속변수군
target_col = 'target_maxrise_6h'

# 독립변수군
feature_cols = [
    col for col in train_full.columns
    if col not in ('timestamp', target_col)
]

assert len(feature_cols) == 32, (
    f'독립변수 개수 이상: 예상 32개, 현재 {len(feature_cols)}개'
)

## 2. 이상치 제거

In [ ]:
# ==============================================================
# 2-1. 이상치(계측오류) 제거: train_full 종속변수 이상치 제거(1개)
# ==============================================================
# 변수별 분포특성 확인단계에서 확인됨

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 이상치 전후 시계열 확인: 전후 시계열에서 종속변수 -8.74는 이상치로 판단
cond_maxrise_out_st1 = train_full['station_id'] == 'ST0007'
cond_maxrise_out_time = train_full['timestamp'].between(
    '2103-02-27 03:00', '2103-02-28 00:00'
)
print('==================== 이상치 전후 시계열 ====================')
display(
    train_full.loc[
        (cond_maxrise_out_st1 & cond_maxrise_out_time),
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
)
print()

# 이상치 조건설정
cond_maxrise_out = (
    cond_maxrise_out_st1 &
    cond_maxrise_out_time &
    (train_full['target_maxrise_6h'] < -8)
)
                
print(f"이상치 개수(제거대상): {cond_maxrise_out.sum()}")
print(f"이상치 행 제거 전 데이터 개수: {len(train_full)}") 
assert cond_maxrise_out.sum() == 1, "이상치 제거 대상 개수 확인 필요"

# 이상치 제거
train_full = train_full.drop(
    index = train_full.loc[cond_maxrise_out].index
)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train_full)}")

In [ ]:
# ======================================================================
# 2-2. 이상치(계측오류) 제거: train_full 수위변수 이상치 제거(ST0445, 12개)
# ======================================================================
# 변수별 분포특성 확인단계에서 확인됨

# 이상치 전후 시계열 확인: 전후 시계열에서 ST0445 관측소는의 수위 17.63은 비정상적 상승으로 이상치로 판단
cond_wl_out_st1 = train_full['station_id'] == 'ST0445'
cond_wl_out_time1 = (
    train_full['timestamp'].between('2100-08-09 00:00:00', '2100-08-10 00:00:00') |
    train_full['timestamp'].between('2102-09-17 00:00:00', '2102-09-19 00:00:00') |
    train_full['timestamp'].between('2101-11-02 00:00:00', '2101-11-03 00:00:00')
)
print('==================== 이상치 전후 시계열 ====================')
display(
    train_full.loc[
        (cond_wl_out_st1 & cond_wl_out_time1),
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
)
print()

# 이상치 조건설정
cond_wl_out1 = (
    cond_wl_out_st1 &
    cond_wl_out_time1 &
    ((train_full[wl_cols] > 17).any(axis=1))
)
                
print(f"이상치 개수(제거대상): {cond_wl_out1.sum()}")
print(f"이상치 행 제거 전 데이터 개수: {len(train_full)}") 
assert cond_wl_out1.sum() == 12, "이상치 제거 대상 개수 확인 필요"

# 이상치 제거
train_full = train_full.drop(
    index = train_full.loc[cond_wl_out1].index
)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train_full)}")

In [ ]:
# ======================================================================
# 2-3. 이상치(계측오류) 제거: train_full 수위변수 이상치 제거(ST0066, 20개)
# ======================================================================
# 변수별 분포특성 확인단계에서 확인됨

# 이상치 관측소
cond_wl_out_st2 = train_full['station_id'] == 'ST0066'

# ST0066 관측소의 현재 수위(wl_t) 분위수
# 0.99분위수가 23.55, 0.98분위수가 12.45 -> 23 이상은 계측오류로 판단
print('=============== ST0066관측소 현재수위 분위수 ===============')
display(
    train_full.loc[
        cond_wl_out_st2, 'wl_t'
    ].quantile([0.5, 0.9, 0.95, 0.98, 0.99, 1.00])
)
print()

# 이상치 전후 시계열 확인
cond_wl_out_time2 = (
    train_full['timestamp'].between('2100-08-19 15:00:00', '2100-08-20 00:00:00') |
    train_full['timestamp'].between('2102-03-31 00:00:00', '2102-04-01 03:00:00')
)

print('==================== 이상치 전후 시계열 ====================')
display(
    train_full.loc[
        (cond_wl_out_st2 & cond_wl_out_time2),
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
)
print()

# 이상치 조건설정
cond_wl_out2 = (
    cond_wl_out_st2 &
    ((train_full[wl_cols] > 23).any(axis=1))
)
                
print(f"이상치 개수(제거대상): {cond_wl_out2.sum()}")
print(f"이상치 행 제거 전 데이터 개수: {len(train_full)}") 
assert cond_wl_out2.sum() == 20, "이상치 제거 대상 개수 확인 필요"

# 이상치 제거
train_full = train_full.drop(
    index = train_full.loc[cond_wl_out2].index
)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train_full)}")

In [ ]:
# ==========================================================================
# 2-4. 이상치(계측오류) 제거: train_full 수위변수 이상치 제거(4개 관측소, 15개)
# ==========================================================================
# train, test셋 분포 비교에서 확인
# 전후 시계열을 확인하여 이상계측 임계치를 설정하여 이상치 제거

# ST0089관측소 이상치 전후 시계열 확인
cond_wl_out_st3 = train_full['station_id'] == 'ST0089'
cond_wl_out_time3 = (
    train_full['timestamp'].between('2102-01-01 00:00', '2102-01-02 00:00')
)
print('==================== ST0089관측소 이상치 전후 시계열 ====================')
display(
    train_full.loc[
        (cond_wl_out_st3 & cond_wl_out_time3),
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
)
print()
# ST0089관측소 이상치 조건 설정
cond_wl_out3 = (
    cond_wl_out_st3 &
    cond_wl_out_time3 &
    ((train_full[wl_cols] > 6).any(axis=1))
)


# ST0158관측소 이상치 전후 시계열 확인
cond_wl_out_st4 = train_full['station_id'] == 'ST0158'
cond_wl_out_time4 = (
    train_full['timestamp'].between('2102-05-10 0:00', '2102-05-12 00:00')
)
print('==================== ST0158관측소 이상치 전후 시계열 ====================')
display(
    train_full.loc[
        (cond_wl_out_st4 & cond_wl_out_time4),
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
)
print()
# ST0158관측소 이상치 조건 설정
cond_wl_out4 = (
    cond_wl_out_st4 &
    cond_wl_out_time4 &
    ((train_full[wl_cols] > 7).any(axis=1))
)


# ST0007관측소 이상치 전후 시계열 확인
cond_wl_out_st5 = train_full['station_id'] == 'ST0007'
cond_wl_out_time5 = (
    train_full['timestamp'].between('2103-12-01 00:00', '2103-12-02 02:00')
)
print('==================== ST0007관측소 이상치 전후 시계열 ====================')
display(
    train_full.loc[
        (cond_wl_out_st5 & cond_wl_out_time5),
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
)
print()
# ST0007관측소 이상치 조건 설정
cond_wl_out5 = (
    cond_wl_out_st5 &
    cond_wl_out_time5 &
    ((train_full[wl_cols] > 8).any(axis=1))
)


# ST0448관측소 이상치 전후 시계열 확인
cond_wl_out_st6 = train_full['station_id'] == 'ST0448'
cond_wl_out_time6 = (
    train_full['timestamp'].between('2103-12-02 00:00', '2103-12-03 00:00')
)
print('==================== ST0448관측소 이상치 전후 시계열 ====================')
display(
    train_full.loc[
        (cond_wl_out_st6 & cond_wl_out_time6),
        ['timestamp', 'station_id'] + wl_cols + [target_col]
    ]
)
print()
# ST0448관측소 이상치 조건 설정
cond_wl_out6 = (
    cond_wl_out_st6 &
    cond_wl_out_time6 &
    ((train_full[wl_cols] > 9).any(axis=1))
)

                
print(
    f"이상치 개수(제거대상): {
        (cond_wl_out3 | cond_wl_out4 | cond_wl_out5 | cond_wl_out6)
        .sum()
    }"
)
print(f"이상치 행 제거 전 데이터 개수: {len(train_full)}") 
assert (cond_wl_out3 | cond_wl_out4 | cond_wl_out5 | cond_wl_out6).sum() == 15, "이상치 제거 대상 개수 확인 필요"

# 이상치 제거
train_full = train_full.drop(
    index = train_full.loc[
        (cond_wl_out3 | cond_wl_out4 | cond_wl_out5 | cond_wl_out6)
    ].index
)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train_full)}")

## 3. 강우사상 라벨링, GroupKFold 5-Fold 배정(CSV로 저장)

In [ ]:
# ============================================================
# 3-1. 강우사상별 데이터 분할 -> event_id 컬럼 생성
# ============================================================

# 전체 timestamp 추출(중복없이), 시간순 정렬
event_time = (
    train_full[['timestamp']] 
    .drop_duplicates() 
    .sort_values('timestamp') 
    .reset_index(drop = True) 
)

# 차분
event_time['event_time_diff'] = (
    event_time['timestamp']
    .diff()
)

# 차분값이 1시간이 아닌 경우를 event 경계로 추출
cond_split_time = (
    event_time['event_time_diff'] != pd.Timedelta(hours=1)
)

# 강우사상 시간대별 event_id 부여
event_time['event_id'] = cond_split_time.cumsum()

# 시험처가 제시한 강우사상 개수와 맞는지 확인: 122개
assert event_time['event_id'].nunique() == 122, (
    f"주최측이 제시한 강우사상 122개, 현재 {event_time['event_id'].nunique()}개, 오류"
)

# train_full에 event_id 컬럼 붙이기(매핑)
rule_event_id_map = event_time.set_index('timestamp')['event_id']  # 매핑규칙

train_full['event_id'] = train_full['timestamp'].map(rule_event_id_map)

assert not train_full['event_id'].isnull().any(), (
    'event_id 매핑 누락 발생'
)

In [ ]:
# ============================================================
# 3-2. GroupKFold 5-Fold 배정, csv 저장
# ============================================================

from sklearn.model_selection import GroupKFold

# ------------------------------------------------------------
# Group 설정: 강우사상(event_id) 단위로 fold 배정
# ------------------------------------------------------------

kf_groups = train_full['event_id']

print('<시간사상 Group 확인>')
print(f'전체 학습 행 수   : {len(train_full):,}')
print(f'시간사상 개수     : {kf_groups.nunique():,}')
print()

assert not kf_groups.isnull().any(), 'event_id 결측치 존재'


# ------------------------------------------------------------
# GroupKFold 정의
# ------------------------------------------------------------

N_SPLITS = 5

gkf = GroupKFold(n_splits=N_SPLITS)
# shuffle=False(기본값), Fold별 행 개수가 비슷하게 배정

# 시간사상별 Fold 번호 저장용 데이터프레임
kf_fold_f = pd.DataFrame({
    'event_id': kf_groups.unique()
})

kf_fold_f['fold_num'] = 0  # Fold 번호 컬럼 생성


# ------------------------------------------------------------
# Fold 배정 및 검증
# ------------------------------------------------------------

for fold, (train_idx, valid_idx) in enumerate(  # 반환값: fold 번호, (해당폴드 train 인덱스, val 인덱스)
    gkf.split(train_full, groups=kf_groups),  # 5개 fold 배정 결정
    start=1  # fold 1부터 시작
):
    # Fold별 train, val 사상 중복 여부 확인
    train_groups = set(kf_groups.iloc[train_idx])
    valid_groups = set(kf_groups.iloc[valid_idx])
    overlap = train_groups & valid_groups
    assert len(overlap) == 0, f'Fold {fold}: train, val 간에 사상 중복'

    # 검증 Fold에 해당하는 시간사상에 Fold 번호 부여
    kf_fold_f.loc[
        kf_fold_f['event_id'].isin(valid_groups),  # val그룹에 속하는 event_id는
        'fold_num'                                 # fold_num 컬럼에
    ] = fold                                       # 현재 fold 번호를 기재

    print(
        f'Fold {fold} | '
        f'train={len(train_idx):,} | '  # 인덱스 개수(행 개수)
        f'valid={len(valid_idx):,} | '
        f'train events={len(train_groups):,} | '  # event_id 개수
        f'valid events={len(valid_groups):,} | '
        f'event overlap={len(overlap)}'
    )


# ------------------------------------------------------------
# 3-4. Fold 배정 결과 최종 검증
# ------------------------------------------------------------

assert kf_fold_f['event_id'].is_unique, 'event_id 중복'
assert kf_fold_f['fold_num'].between(1, N_SPLITS).all(), 'Fold 배정 누락'


# ------------------------------------------------------------
# 3-5. Fold 배정 CSV 저장
# ------------------------------------------------------------

kf_fold_f.to_csv(
    PROJECT_ROOT / 'data' / '0_kf_train_full_match.csv',
    index=False
)

print('\nFold 배정 CSV 저장 완료')


In [ ]:
# ============================================================
# 3-4. 최종 train_full, test 저장
# ============================================================

train_full.to_parquet(PROJECT_ROOT / 'data' / '2_train_full_fe.parquet')
test.to_parquet(PROJECT_ROOT / 'data' / '2_test_fe.parquet')